In [ ]:
from sortedcontainers import SortedDict

class Opentable:
    def __init__(self):
        self.opentablelist = list()
        
    def get(self):
        return self.opentablelist
        
    def add(self, side, price, amount, ts):
        todel = list()
        ii = 0
        profit = 0
        
        left_amount = amount
        for i in self.opentablelist:
            if(i["side"] == "bid"):
                ii+=1
                continue
                
            if(left_amount >= i["amount"]):
                profit += (((i["price"] - price)/i["price"] * i["amount"]))
                todel.append(ii)
                left_amount-=i["amount"]
            elif(left_amount < i["amount"]):
                profit += (((i["price"] - price)/i["price"] * left_amount))
                self.opentablelist[ii]["amount"] = i["amount"] - left_amount
                left_amount = 0

                
            ii+=1
        if(left_amount > 0):
            self.opentablelist.append({
                "price":price,
                "amount":left_amount,
                "side":side,
                "ts":ts,
            }) 
        ap = 0
        for i in todel:
            
            del(self.opentablelist[i-ap])
            ap+=1
            
        return profit
    
 
                    
    def drop(self, side, price, amount, ts):
        todel = list()
        ii = 0
        profit = 0
       
        
        
        left_amount = amount
        for i in self.opentablelist:
            if(i["side"] == "ask"):
                ii+=1
                continue
                
            if(left_amount >= i["amount"]):
                profit += (((price - i["price"])/i["price"] * i["amount"]))
                todel.append(ii)
                left_amount-=i["amount"]
            elif(left_amount < i["amount"]):
                profit += (((price - i["price"])/i["price"] * left_amount))
                self.opentablelist[ii]["amount"] = i["amount"] - left_amount
                left_amount = 0

       
            ii+=1
        if(left_amount > 0):
            self.opentablelist.append({
                "price":price,
                "amount":left_amount,
                "side":side,
                "ts":ts,
            }) 
        ap = 0
        for i in todel:
            del(self.opentablelist[i-ap])
            ap+=1
        return profit
    
    def get_openrisk(self):
        total = 0
        lbids = 1000000
        hasks = 0
        for i in self.opentablelist:
            if(i["side"] == "bid"):
                total+=i["amount"]
                if(i["price"]<lbids):lbids = i["price"]
            elif(i["side"] == "ask"):
                total-=i["amount"]
                if(i["price"]>hasks):hasks = i["price"]
                
        lbids = None if lbids == 1000000 else lbids
        hasks = None if hasks == 0 else hasks
        return total,lbids,hasks
    
    def get_float_profit(self, bid0, ask0):
        profit = 0.
        for i in self.opentablelist:
            if(i["side"] == "bid"):
                profit += ((ask0 - i["price"]) / i["price"] * i["amount"])
            elif(i["side"] == "ask"):
                profit += ((i["price"] - bid0) / i["price"] * i["amount"])
        return profit

In [ ]:
import hashlib
import hmac
import time
import requests
import logging
import pandas as pd
from urllib.parse import urlencode
import warnings
warnings.filterwarnings('ignore')

class binance_rest_client(object):
    def __init__(self, apikey, secret, host='https://papi.binance.com'):
        self.apikey = apikey
        self.secret = secret
        self.host = host
        self._l = logging

    def __hashing(self, query_string):
        return hmac.new(self.secret.encode('utf-8'), query_string.encode('utf-8'), hashlib.sha256).hexdigest()

    def __dispatch_request(self, http_method):
        session = requests.Session()
        session.headers.update({
            'Content-Type': 'application/json',
            'X-MBX-APIKEY': self.apikey
        })
        return {
            'GET': session.get,
            'DELETE': session.delete,
            'PUT': session.put,
            'POST': session.post,
        }.get(http_method, 'GET')

    def http_request(self, http_method, url_path, payload={}, signed=True):
        query_string = urlencode(payload, True)
        print(query_string)
        if signed:
            if query_string:
                query_string = "{}&timestamp={}".format(query_string, int(time.time() * 1000))
            else:
                query_string = 'timestamp={}'.format(int(time.time() * 1000))

            url = self.host + url_path + '?' + query_string + '&signature=' + self.__hashing(query_string)
        else:
            url = self.host + url_path

        # self._l.info("{} {}".format(http_method, url))
        params = {'url': url, 'params': {}, 'timeout': 5}
        try:
            response = self.__dispatch_request(http_method)(**params)

            if response.status_code == 200:
                return response.json()
            elif response.status_code == 400:
                res = response.json()
                return self._response(status='error', msg=res['msg'], code=res['code'])
            if response.status_code == 418:
                time.sleep(3)
            elif response.status_code == 429:
                return self._response(status='error',
                                      msg='Repeatedly violating rate limits and/or failing to back off after receiving 429s will result in an automated IP ban',
                                      code=429)
        except Exception as e:
            self._l.error("request error: %s" % e)
            return self._response(status='error', msg=str(e))

    def _response(self, status, msg='', data=None, code=0):
        return {'status': status, 'code': code, 'data': data, 'msg': msg}


binance_client = binance_rest_client("z9rb55Ix9vyBvzHjYVxZT0mRshd9Qyvvs40vKoCZPhb7FWwAbiUVVsdfQXv5AHHZ","yVYqipzjCe7S7Eydc4kAEMbEvi68Qmo2un91CAm0lOIXaY5vdoXf4W3YW5dNYkYw")

In [ ]:
import os,warnings,time 
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
%run opentable.ipynb 
%run binance_loan.ipynb 

warnings.filterwarnings("ignore") 
plt.rcParams["figure.figsize"] = (20,3) 

base_path = "./orders/" 
fls = os.listdir(base_path) 

# st_sec = 1724112000 * 1000 # strategy start 
# st_sec = 1724929200 * 1000 # liang_torch start 
# st_sec = 1729854000 * 1000 
st_sec = 1733011200 * 1000 
ed_sec = 2723773600 * 1000 


ks_file = [
    # {"exchange":"binanceswap", "ttype":"maker", "fee":-0.00005, "sid":0},
    {"exchange":"binanceswap", "ttype":"maker", "fee":0.0, "sid":0},
    {"exchange":"binanceswap", "ttype":"taker", "fee":0.000144, "sid":0},
    {"exchange":"binancespot", "ttype":"maker", "fee":0.0, "sid":1},
    {"exchange":"binancespot", "ttype":"taker", "fee":0.0001725, "sid":1},
] 
kskeys = [] 
for ks in ks_file: 
    if ks["exchange"] not in kskeys: 
        kskeys.append(ks["exchange"]) 
nks = pd.DataFrame(ks_file) 
nks.columns = ["key",  "fee", "sid", "ttype"] 

ori_nrr = None 

for fl in fls: 
    if fl.startswith("2025-02") | fl.startswith("2025-01"): 
        print(fl) 
        fl_df = pd.read_csv(base_path + fl,header=None) 
        if len(fl_df.columns) == 20:
            fl_df.columns = ["client_order_id", "ttype", "exchange", "etype", "symbol", "order_status", "price", "side", "amount", "filled_amount", "contract_value", "last_filled_amount", "create_ts", "last_update_ts", "next_check_ts", "n_nexist", "last_cancel_ts", "from_key","posu","openrisku"]
        else:
            fl_df.columns = ["client_order_id", "ttype", "exchange", "etype", "symbol", "order_status", "price", "side", "amount", "filled_amount", "contract_value", "last_filled_amount", "create_ts", "last_update_ts", "next_check_ts", "n_nexist", "last_cancel_ts", "from_key"] 

        fl_df["amount"] = fl_df["amount"] * fl_df["contract_value"]
        fl_df["filled_amount"] = fl_df["filled_amount"] * fl_df["contract_value"] 
        fl_df_grouped = fl_df.groupby('client_order_id').tail(1) 
        fl_df_grouped["key"] = fl_df_grouped["exchange"] + fl_df_grouped["etype"] 
        fl_df_grouped["last"] = fl_df_grouped["last_update_ts"] - fl_df_grouped["create_ts"] 
        t_df = fl_df_grouped[["symbol","ttype","key","last","price","side","filled_amount","last_update_ts","create_ts","order_status","from_key","client_order_id","posu","openrisku"]] 
        t_df.columns = ["symbol","ttype","key","last","price","side","amount","luts","ts","status","from_key","client_order_id","posu","openrisku"] 
        ori_nrr = t_df if ori_nrr is None else pd.concat([ori_nrr,t_df]) 
    
ori_nrr = ori_nrr[(ori_nrr["ts"] >= st_sec) & (ori_nrr["ts"] < ed_sec)] 
ori_nrr = ori_nrr.sort_values("ts",ascending=True) 
nrr = ori_nrr[ori_nrr["amount"] > 0] 
if st_sec == 0: 
    st_sec = nrr["ts"].min() 
symbols = nrr["symbol"].unique() 

nrr = pd.merge(nrr, nks, on=["key", "ttype"], how="left") 
nrr["amountu"] = nrr["amount"] * nrr["price"] 
nrr["fees"] = nrr["fee"] * nrr["amountu"] 
nrr.sort_values("ts", inplace=True) 
print(len(ori_nrr),len(nrr)) 
nrr[:2] 


ori_nrr["dt"] = pd.to_datetime(ori_nrr["ts"]+ 8*3600*1000, unit='ms') 
ori_nrr.plot("dt","last") 
nrr['amountu'].sum() 



alls = {} 
for symbol in symbols: 
    pfix = 0 
    ot = Opentable() 
    ns = nrr[nrr["symbol"] == symbol].sort_values("ts") 
    
    idx = 0 
    for i, row in ns.iterrows(): 
        fee = row.fee 
        if idx == 0: 
            if(row.openrisku > 0): 
                _ = ot.add("bid", row.price, abs(row.openrisku), row.ts) 
            else:
                _ = ot.drop("ask", row.price, abs(row.openrisku), row.ts) 
        idx += 1 
        if(row.side == "buy"):
            p = ot.add("bid", row.price, row.price * row.amount, row.ts) 
            pfix+=p 
            pfix-=row.fees 
        elif(row.side == "sell"): 
            p = ot.drop("ask", row.price, row.price * row.amount, row.ts) 
            pfix+=p 
            pfix-=row.fees 
        pfloat = ot.get_float_profit(row.price,row.price) 
        openrisk,_l,_l = ot.get_openrisk() 
        if int(row.ts/1000) not in alls:
            alls[int(row.ts/1000)] = {} 
        alls[int(row.ts/1000)][symbol] = pfix+pfloat 


ass = pd.DataFrame(alls).T.reset_index()

################
mst_sec = (int(ass["index"].min()/3600)+1)*3600 
med_sec = int(time.time()/3600)*3600 
assindex_list = list(ass["index"].unique()) 
while mst_sec <= med_sec: 
    if mst_sec not in assindex_list: 
        new_row = {'index': mst_sec} 
        for col in ass.columns:
            if col == "index":
                continue
            new_row[col] = np.nan
        ass = ass.append(new_row, ignore_index=True)
    mst_sec += 3600
ass = ass.sort_values("index",ascending=True) 
ass = ass.reset_index(drop=True) 

ass = ass.fillna(method="ffill") 
ass["ss"] = ass[symbols].sum(axis=1) 
ass["dt"] = pd.to_datetime(ass["index"]+8*3600, unit="s") 
ass.plot(x="dt", y="ss") 
plt.title("total") 